# SpatialDataAgent-DB starter pack: five minutes, no login

88 representative spatial transcriptomics samples (12 platforms, 20 organ groups, 60 with tissue images),
byte-identical to the full SpatialDataAgent-DB release. Nothing here needs a Hugging Face account.

In [ ]:
!pip -q install "spatial-data-agent-db[analysis] @ git+https://github.com/Biogod2020/spatial-data-agent-db" matplotlib

## 1. What is in the pack?

In [ ]:
import sda_db

starter = sda_db.open_catalog("starter")
print(starter)
print(starter.summary(by="platform")["groups"])
starter.to_pandas()[["gsm", "platform", "organism", "tissue", "n_obs", "image_stain", "split"]].head(10)

## 2. Load one H&E sample and look at it

In [ ]:
he = starter.search(image_stain="H&E", organism="human", sort="image_um_per_px")   # sharpest human H&E first
gsm = he.gsms[0]
print(gsm, he.get(gsm)["tissue"], he.get(gsm)["study_title"])
adata = sda_db.load(gsm, catalog=starter, include_image=True)   # downloads ~ tens of MB, checks SHA256
adata

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from sda_db.image import read_image, read_patch, to_array_xy
from sda_db.io import image_coordinates

meta = json.loads(adata.uns["sda_db"]["sample_json"])
directory = adata.uns["sda_db"]["directory"]
xy = image_coordinates(adata, meta)                                  # level-0 image pixel of every spot
image, info = read_image(directory, level="auto", max_size=1600)
spots = to_array_xy(xy, info)
names = [n.upper() for n in adata.var["feature_name"]]
markers = [g for g in ["KRT19", "EPCAM", "COL1A1", "PTPRC", "MBP", "SFTPC", "ALB", "MYH6"] if g in names]
totals = {g: float(adata[:, names.index(g)].X.sum()) for g in markers}
gene = max(totals, key=totals.get)                                     # the most abundant of a few markers
values = np.log1p(np.asarray(adata[:, names.index(gene)].X.todense()).ravel())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image); axes[0].set_title("H&E (image.tif)")
axes[1].imshow(image, alpha=0.4); axes[1].scatter(*spots.T, c=values, s=8, cmap="Blues"); axes[1].set_title(f"{gene} per spot")
patch, _ = read_patch(directory, xy[int(values.argmax())], 224)
axes[2].imshow(patch); axes[2].set_title("224 px patch at the top spot")
for ax in axes: ax.axis("off")

## 3. Find samples by gene, without downloading them

In [ ]:
panels = starter.search(genes=["EPCAM"], where="n_vars<=10000")      # targeted panels that measure EPCAM
panels.to_pandas()[["gsm", "platform", "tissue", "n_vars"]]

## 4. The same, for the full atlas

The paper release has 12,163 samples. Searching it also needs no login:
`sda_db.open_catalog().search("alzheimer hippocampus", organism="human")`.
Downloading its files needs approved access: https://huggingface.co/datasets/Biogod/SpatialDataAgent-DB